In [ ]:
# %% [markdown]
# # Anomaly Detection Visualizations
# 
# This notebook provides advanced visualizations for anomaly detection in financial fraud detection.

# %% [markdown]
# ## 1. Setup and Configuration

# %%
import warnings
warnings.filterwarnings('ignore')

# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
import yaml
from pathlib import Path
import sys

# Visualization imports
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import matplotlib.cm as cm
from matplotlib.patches import Ellipse
import matplotlib.patches as mpatches

# ML imports
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import DBSCAN, KMeans
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope
from scipy import stats
from scipy.spatial.distance import mahalanobis

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
np.random.seed(42)

print("All libraries imported successfully!")

# %% [markdown]
# ## 2. Load and Prepare Data

# %%
# Load configuration
config_path = '../config/settings.yaml'
try:
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    print("Configuration loaded successfully!")
except:
    config = {}

# Load data
data_path = '../data/processed/transactions_processed.csv'
if Path(data_path).exists():
    df = pd.read_csv(data_path)
    print(f"Data loaded from {data_path}")
else:
    # Generate synthetic data
    print("Generating synthetic data...")
    np.random.seed(42)
    n_samples = 5000
    
    data = {
        'transaction_id': [f'TX{i:08d}' for i in range(n_samples)],
        'customer_id': [f'CUST{np.random.randint(10000, 99999)}' for _ in range(n_samples)],
        'amount': np.random.exponential(100, n_samples).clip(1, 10000),
        'transaction_type': np.random.choice(['purchase', 'withdrawal', 'transfer'], n_samples),
        'merchant_category': np.random.choice(['retail', 'food', 'travel'], n_samples),
        'location': np.random.choice(['US', 'UK', 'DE', 'FR'], n_samples),
        'device_type': np.random.choice(['mobile', 'desktop'], n_samples),
        'hour_of_day': np.random.randint(0, 24, n_samples),
        'day_of_week': np.random.randint(0, 7, n_samples),
        'timestamp': pd.date_range(start='2024-01-01', periods=n_samples, freq='H'),
        'is_fraud': np.random.choice([0, 1], n_samples, p=[0.97, 0.03])
    }
    
    df = pd.DataFrame(data)
    
    # Make fraud transactions different
    fraud_mask = df['is_fraud'] == 1
    df.loc[fraud_mask, 'amount'] = np.random.exponential(500, fraud_mask.sum()).clip(100, 20000)
    df.loc[fraud_mask, 'hour_of_day'] = np.random.choice([2, 3, 4, 5], fraud_mask.sum())
    
    # Save for future use
    df.to_csv('../data/processed/transactions_processed.csv', index=False)

print(f"Data shape: {df.shape}")
print(f"Fraud rate: {df['is_fraud'].mean()*100:.2f}%")

# Prepare features
def prepare_features(df):
    """Prepare features for anomaly detection"""
    df_features = df.copy()
    
    # Drop non-numeric columns
    columns_to_drop = ['transaction_id', 'timestamp', 'customer_id']
    columns_to_drop = [col for col in columns_to_drop if col in df_features.columns]
    df_features = df_features.drop(columns=columns_to_drop, errors='ignore')
    
    # One-hot encode categorical columns
    categorical_cols = df_features.select_dtypes(include=['object']).columns
    if len(categorical_cols) > 0:
        df_features = pd.get_dummies(df_features, columns=categorical_cols, drop_first=True)
    
    # Handle missing values
    df_features = df_features.fillna(0)
    
    # Separate features and target
    if 'is_fraud' in df_features.columns:
        y = df_features['is_fraud']
        X = df_features.drop(columns=['is_fraud'])
    else:
        X = df_features
        y = None
    
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    return X_scaled, y, X.columns.tolist()

X_scaled, y, feature_names = prepare_features(df)

print(f"Features prepared: {X_scaled.shape}")
print(f"Number of features: {len(feature_names)}")

# %% [markdown]
# ## 3. Dimensionality Reduction for Visualization

# %%
class DimensionalityReducer:
    """Reduce dimensionality for visualization"""
    
    def __init__(self, X, y=None):
        self.X = X
        self.y = y
        
    def apply_pca(self, n_components=2):
        """Apply PCA for dimensionality reduction"""
        from sklearn.decomposition import PCA
        
        pca = PCA(n_components=n_components, random_state=42)
        X_pca = pca.fit_transform(self.X)
        
        # Calculate explained variance
        explained_variance = pca.explained_variance_ratio_
        cumulative_variance = np.cumsum(explained_variance)
        
        print(f"PCA Results:")
        print(f"  Explained variance by component: {explained_variance}")
        print(f"  Cumulative explained variance: {cumulative_variance}")
        
        return X_pca, pca, explained_variance
    
    def apply_tsne(self, n_components=2, perplexity=30):
        """Apply t-SNE for dimensionality reduction"""
        tsne = TSNE(n_components=n_components, 
                   perplexity=perplexity,
                   random_state=42,
                   init='pca',
                   learning_rate='auto')
        
        X_tsne = tsne.fit_transform(self.X)
        
        print(f"t-SNE applied with perplexity={perplexity}")
        
        return X_tsne, tsne
    
    def apply_umap(self, n_components=2):
        """Apply UMAP for dimensionality reduction"""
        try:
            import umap
            
            reducer = umap.UMAP(n_components=n_components,
                               random_state=42,
                               n_neighbors=15,
                               min_dist=0.1)
            
            X_umap = reducer.fit_transform(self.X)
            
            print("UMAP applied successfully")
            
            return X_umap, reducer
            
        except ImportError:
            print("UMAP not installed. Install with: pip install umap-learn")
            return None, None
    
    def visualize_reduction(self, X_reduced, method='PCA', y=None):
        """Visualize dimensionality reduction results"""
        
        if X_reduced.shape[1] not in [2, 3]:
            print("Can only visualize 2D or 3D reductions")
            return
        
        if y is None:
            y = self.y if self.y is not None else np.zeros(len(X_reduced))
        
        if X_reduced.shape[1] == 2:
            # 2D visualization
            fig, axes = plt.subplots(1, 2, figsize=(15, 6))
            
            # Scatter plot
            scatter = axes[0].scatter(X_reduced[:, 0], X_reduced[:, 1], 
                                     c=y, cmap='viridis', alpha=0.6)
            axes[0].set_xlabel(f'{method} Component 1')
            axes[0].set_ylabel(f'{method} Component 2')
            axes[0].set_title(f'{method} 2D Projection')
            plt.colorbar(scatter, ax=axes[0], label='Fraud Indicator')
            
            # Density plot
            if len(np.unique(y)) == 2:
                for label in [0, 1]:
                    mask = y == label
                    axes[1].scatter(X_reduced[mask, 0], X_reduced[mask, 1], 
                                   alpha=0.5, label=f'Class {label}')
                axes[1].legend()
            else:
                axes[1].hist2d(X_reduced[:, 0], X_reduced[:, 1], bins=50, cmap='hot')
            
            axes[1].set_xlabel(f'{method} Component 1')
            axes[1].set_ylabel(f'{method} Component 2')
            axes[1].set_title(f'{method} Density Plot')
            
        else:  # 3D
            fig = plt.figure(figsize=(12, 8))
            ax = fig.add_subplot(111, projection='3d')
            
            scatter = ax.scatter(X_reduced[:, 0], X_reduced[:, 1], X_reduced[:, 2],
                               c=y, cmap='viridis', alpha=0.6)
            
            ax.set_xlabel(f'{method} Component 1')
            ax.set_ylabel(f'{method} Component 2')
            ax.set_zlabel(f'{method} Component 3')
            ax.set_title(f'{method} 3D Projection')
            
            plt.colorbar(scatter, label='Fraud Indicator')
        
        plt.tight_layout()
        plt.show()
        
        # Interactive Plotly visualization
        self._interactive_visualization(X_reduced, y, method)
    
    def _interactive_visualization(self, X_reduced, y, method):
        """Interactive visualization with Plotly"""
        
        if X_reduced.shape[1] == 2:
            fig = px.scatter(
                x=X_reduced[:, 0],
                y=X_reduced[:, 1],
                color=y.astype(str) if y is not None else None,
                title=f'{method} 2D Projection - Interactive',
                labels={'x': f'{method} Component 1', 'y': f'{method} Component 2'},
                opacity=0.7
            )
            
            if y is not None:
                fig.update_traces(
                    hovertemplate='<br>'.join([
                        f'{method}1: %{{x:.3f}}',
                        f'{method}2: %{{y:.3f}}',
                        'Class: %{marker.color}'
                    ])
                )
            
        else:  # 3D
            fig = px.scatter_3d(
                x=X_reduced[:, 0],
                y=X_reduced[:, 1],
                z=X_reduced[:, 2],
                color=y.astype(str) if y is not None else None,
                title=f'{method} 3D Projection - Interactive',
                labels={'x': f'{method} Component 1', 
                       'y': f'{method} Component 2',
                       'z': f'{method} Component 3'},
                opacity=0.7
            )
        
        fig.update_layout(
            hovermode='closest',
            showlegend=True if y is not None else False
        )
        
        fig.show()

# %%
# Apply dimensionality reduction
reducer = DimensionalityReducer(X_scaled, y)

# PCA
print("\n🔍 Applying PCA...")
X_pca, pca, explained_variance = reducer.apply_pca(n_components=3)
reducer.visualize_reduction(X_pca, method='PCA')

# Plot explained variance
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scree plot
axes[0].plot(range(1, len(explained_variance) + 1), explained_variance, 'o-')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot')
axes[0].grid(True, alpha=0.3)

# Cumulative variance
axes[1].plot(range(1, len(explained_variance) + 1), np.cumsum(explained_variance), 'o-')
axes[1].axhline(y=0.95, color='r', linestyle='--', alpha=0.5, label='95% threshold')
axes[1].axhline(y=0.90, color='g', linestyle='--', alpha=0.5, label='90% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# t-SNE
print("\n🔍 Applying t-SNE...")
X_tsne, tsne = reducer.apply_tsne(n_components=2, perplexity=30)
reducer.visualize_reduction(X_tsne, method='t-SNE')

# Try different perplexity values
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

perplexities = [5, 15, 30, 50]

for idx, perplexity in enumerate(perplexities):
    tsne_temp = TSNE(n_components=2, perplexity=perplexity, random_state=42)
    X_tsne_temp = tsne_temp.fit_transform(X_scaled[:1000])  # Sample for speed
    
    scatter = axes[idx].scatter(X_tsne_temp[:, 0], X_tsne_temp[:, 1], 
                               c=y[:1000] if y is not None else 'blue', 
                               alpha=0.6, cmap='viridis')
    axes[idx].set_title(f't-SNE (perplexity={perplexity})')
    axes[idx].set_xlabel('Component 1')
    axes[idx].set_ylabel('Component 2')
    
    if y is not None:
        plt.colorbar(scatter, ax=axes[idx], label='Fraud Indicator')

plt.tight_layout()
plt.suptitle('t-SNE with Different Perplexity Values', fontsize=14, y=1.02)
plt.show()

# UMAP (if available)
print("\n🔍 Applying UMAP...")
X_umap, umap_reducer = reducer.apply_umap(n_components=2)
if X_umap is not None:
    reducer.visualize_reduction(X_umap, method='UMAP')

# %% [markdown]
# ## 4. Statistical Anomaly Detection Visualizations

# %%
class StatisticalVisualizations:
    """Statistical anomaly detection visualizations"""
    
    def __init__(self, X, feature_names, y=None):
        self.X = X
        self.feature_names = feature_names
        self.y = y
        
    def z_score_analysis(self):
        """Z-score analysis and visualization"""
        
        print("=" * 80)
        print("Z-SCORE ANOMALY DETECTION")
        print("=" * 80)
        
        # Calculate Z-scores
        z_scores = np.abs(stats.zscore(self.X, axis=0))
        
        # Find anomalies (Z-score > 3)
        anomalies = z_scores > 3
        anomaly_counts = anomalies.sum(axis=0)
        total_anomalies = anomalies.sum()
        
        print(f"\n📊 Z-Score Analysis Results:")
        print(f"  Total anomaly flags: {total_anomalies}")
        print(f"  Anomaly rate: {total_anomalies / (self.X.shape[0] * self.X.shape[1]):.4f}")
        
        # Visualize Z-score distributions
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        axes = axes.ravel()
        
        # 1. Z-score distribution for all features
        axes[0].hist(z_scores.flatten(), bins=50, edgecolor='black', alpha=0.7)
        axes[0].axvline(x=3, color='red', linestyle='--', label='Z=3 threshold')
        axes[0].set_xlabel('Absolute Z-score')
        axes[0].set_ylabel('Frequency')
        axes[0].set_title('Z-score Distribution (All Features)')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # 2. Top features with most anomalies
        top_features_idx = np.argsort(anomaly_counts)[-10:][::-1]
        top_features = [self.feature_names[i] for i in top_features_idx]
        
        axes[1].barh(range(len(top_features)), anomaly_counts[top_features_idx])
        axes[1].set_yticks(range(len(top_features)))
        axes[1].set_yticklabels(top_features)
        axes[1].set_xlabel('Number of Anomalies (Z > 3)')
        axes[1].set_title('Top 10 Features with Most Anomalies')
        axes[1].invert_yaxis()
        
        # 3. Correlation between Z-scores and fraud
        if self.y is not None:
            avg_z_scores = z_scores.mean(axis=1)
            
            fraud_z_scores = avg_z_scores[self.y == 1]
            normal_z_scores = avg_z_scores[self.y == 0]
            
            axes[2].hist(normal_z_scores, bins=30, alpha=0.6, label='Normal', color='blue')
            axes[2].hist(fraud_z_scores, bins=30, alpha=0.6, label='Fraud', color='red')
            axes[2].set_xlabel('Average Z-score')
            axes[2].set_ylabel('Frequency')
            axes[2].set_title('Z-score Distribution by Fraud Status')
            axes[2].legend()
            axes[2].grid(True, alpha=0.3)
            
            # Statistical test
            from scipy.stats import ttest_ind
            t_stat, p_value = ttest_ind(fraud_z_scores, normal_z_scores, equal_var=False)
            axes[2].text(0.05, 0.95, f'p-value: {p_value:.4f}', 
                        transform=axes[2].transAxes, fontsize=10,
                        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # 4. Heatmap of anomalies
        if self.X.shape[1] <= 20:  # Limit for readability
            anomaly_matrix = anomalies[:100, :]  # First 100 samples
            axes[3].imshow(anomaly_matrix.T, aspect='auto', cmap='Reds', interpolation='nearest')
            axes[3].set_xlabel('Sample Index')
            axes[3].set_ylabel('Feature')
            axes[3].set_title('Anomaly Heatmap (First 100 Samples)')
            axes[3].set_yticks(range(len(self.feature_names)))
            axes[3].set_yticklabels(self.feature_names, fontsize=8)
        else:
            axes[3].set_visible(False)
        
        plt.tight_layout()
        plt.show()
        
        # Interactive Z-score visualization
        self._interactive_z_score_visualization(z_scores)
        
        return z_scores, anomalies
    
    def _interactive_z_score_visualization(self, z_scores):
        """Interactive Z-score visualization"""
        
        # Create DataFrame for visualization
        z_score_df = pd.DataFrame(z_scores, columns=self.feature_names)
        z_score_df['max_z_score'] = z_score_df.max(axis=1)
        
        if self.y is not None:
            z_score_df['is_fraud'] = self.y
        
        # Interactive scatter plot
        if len(self.feature_names) >= 2:
            fig = px.scatter(
                z_score_df,
                x=self.feature_names[0],
                y=self.feature_names[1],
                color='max_z_score' if self.y is None else 'is_fraud',
                size='max_z_score',
                hover_data=self.feature_names[:5],
                title='Z-score Anomaly Visualization',
                color_continuous_scale='viridis' if self.y is None else None,
                opacity=0.7
            )
            
            # Add Z=3 threshold lines
            fig.add_hline(y=3, line_dash="dash", line_color="red", 
                         annotation_text="Z=3", annotation_position="bottom right")
            fig.add_vline(x=3, line_dash="dash", line_color="red",
                         annotation_text="Z=3", annotation_position="top right")
            
            fig.show()
    
    def iqr_analysis(self):
        """IQR (Interquartile Range) analysis"""
        
        print("\n" + "=" * 80)
        print("IQR ANOMALY DETECTION")
        print("=" * 80)
        
        # Calculate IQR for each feature
        Q1 = np.percentile(self.X, 25, axis=0)
        Q3 = np.percentile(self.X, 75, axis=0)
        IQR = Q3 - Q1
        
        # Calculate bounds
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Find outliers
        outliers = (self.X < lower_bound) | (self.X > upper_bound)
        outlier_counts = outliers.sum(axis=0)
        
        print(f"\n📊 IQR Analysis Results:")
        print(f"  Total outliers detected: {outliers.sum()}")
        
        # Visualize IQR analysis
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        axes = axes.ravel()
        
        # Select a few features to visualize
        if len(self.feature_names) >= 4:
            features_to_show = self.feature_names[:4]
        else:
            features_to_show = self.feature_names
        
        for idx, feature in enumerate(features_to_show):
            if idx < 4:
                feature_idx = self.feature_names.index(feature)
                feature_data = self.X[:, feature_idx]
                
                # Box plot
                bp = axes[idx].boxplot(feature_data, vert=False, patch_artist=True)
                
                # Color outliers
                for i, flier in enumerate(bp['fliers']):
                    flier.set(marker='o', color='red', alpha=0.5)
                
                axes[idx].set_title(f'{feature} - IQR Analysis')
                axes[idx].set_xlabel('Value')
                axes[idx].set_ylabel('Feature')
                axes[idx].grid(True, alpha=0.3)
                
                # Add IQR bounds
                axes[idx].axvline(x=lower_bound[feature_idx], color='green', 
                                linestyle='--', alpha=0.5, label='Lower Bound')
                axes[idx].axvline(x=upper_bound[feature_idx], color='orange', 
                                linestyle='--', alpha=0.5, label='Upper Bound')
                
                if idx == 0:
                    axes[idx].legend()
        
        plt.tight_layout()
        plt.show()
        
        # Interactive violin plot
        if len(features_to_show) > 0:
            fig = go.Figure()
            
            for feature in features_to_show[:5]:  # Limit to 5 features
                feature_idx = self.feature_names.index(feature)
                fig.add_trace(go.Violin(
                    y=self.X[:, feature_idx],
                    name=feature,
                    box_visible=True,
                    meanline_visible=True,
                    points='outliers'
                ))
            
            fig.update_layout(
                title="Violin Plots with Outliers (IQR Method)",
                yaxis_title="Feature Values",
                showlegend=True,
                height=500
            )
            
            fig.show()
        
        return outliers, (lower_bound, upper_bound)
    
    def mahalanobis_distance(self):
        """Mahalanobis distance analysis"""
        
        print("\n" + "=" * 80)
        print("MAHALANOBIS DISTANCE ANALYSIS")
        print("=" * 80)
        
        # Calculate covariance matrix
        cov_matrix = np.cov(self.X.T)
        
        try:
            # Calculate inverse covariance matrix
            inv_cov_matrix = np.linalg.inv(cov_matrix)
            
            # Calculate mean vector
            mean_vector = np.mean(self.X, axis=0)
            
            # Calculate Mahalanobis distance for each point
            mahalanobis_dist = np.zeros(self.X.shape[0])
            
            for i in range(self.X.shape[0]):
                diff = self.X[i] - mean_vector
                mahalanobis_dist[i] = np.sqrt(diff.T @ inv_cov_matrix @ diff)
            
            # Chi-square threshold for p=0.95
            chi2_threshold = stats.chi2.ppf(0.95, df=self.X.shape[1])
            
            # Identify outliers
            outliers = mahalanobis_dist > chi2_threshold
            
            print(f"\n📊 Mahalanobis Distance Analysis:")
            print(f"  Chi-square threshold (p=0.95): {chi2_threshold:.4f}")
            print(f"  Outliers detected: {outliers.sum()} ({outliers.sum()/len(outliers)*100:.2f}%)")
            
            # Visualization
            fig, axes = plt.subplots(2, 2, figsize=(15, 10))
            axes = axes.ravel()
            
            # 1. Mahalanobis distance distribution
            axes[0].hist(mahalanobis_dist, bins=50, edgecolor='black', alpha=0.7)
            axes[0].axvline(x=chi2_threshold, color='red', linestyle='--', 
                           label=f'Chi-square threshold: {chi2_threshold:.2f}')
            axes[0].set_xlabel('Mahalanobis Distance')
            axes[0].set_ylabel('Frequency')
            axes[0].set_title('Mahalanobis Distance Distribution')
            axes[0].legend()
            axes[0].grid(True, alpha=0.3)
            
            # 2. QQ plot against chi-square distribution
            stats.probplot(mahalanobis_dist**2, dist="chi2", 
                          sparams=(self.X.shape[1],), plot=axes[1])
            axes[1].set_title('Q-Q Plot vs Chi-square Distribution')
            axes[1].grid(True, alpha=0.3)
            
            # 3. Scatter plot with outliers highlighted (using PCA)
            if self.X.shape[1] >= 2:
                # Reduce to 2D for visualization
                pca = PCA(n_components=2, random_state=42)
                X_pca = pca.fit_transform(self.X)
                
                scatter = axes[2].scatter(X_pca[:, 0], X_pca[:, 1], 
                                         c=outliers, cmap='coolwarm', alpha=0.6)
                axes[2].set_xlabel('PCA Component 1')
                axes[2].set_ylabel('PCA Component 2')
                axes[2].set_title('PCA Projection with Mahalanobis Outliers')
                axes[2].legend(handles=[
                    mpatches.Patch(color='blue', label='Normal'),
                    mpatches.Patch(color='red', label='Outlier')
                ])
                axes[2].grid(True, alpha=0.3)
            
            # 4. Compare with fraud labels if available
            if self.y is not None:
                fraud_distances = mahalanobis_dist[self.y == 1]
                normal_distances = mahalanobis_dist[self.y == 0]
                
                axes[3].hist(normal_distances, bins=30, alpha=0.6, 
                           label='Normal', color='blue', density=True)
                axes[3].hist(fraud_distances, bins=30, alpha=0.6, 
                           label='Fraud', color='red', density=True)
                axes[3].axvline(x=chi2_threshold, color='green', linestyle='--',
                               label=f'Threshold: {chi2_threshold:.2f}')
                axes[3].set_xlabel('Mahalanobis Distance')
                axes[3].set_ylabel('Density')
                axes[3].set_title('Mahalanobis Distance by Fraud Status')
                axes[3].legend()
                axes[3].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
            
            # Interactive 3D visualization
            if self.X.shape[1] >= 3:
                pca_3d = PCA(n_components=3, random_state=42)
                X_pca_3d = pca_3d.fit_transform(self.X)
                
                fig = px.scatter_3d(
                    x=X_pca_3d[:, 0],
                    y=X_pca_3d[:, 1],
                    z=X_pca_3d[:, 2],
                    color=outliers.astype(str),
                    size=mahalanobis_dist,
                    title='3D Mahalanobis Outlier Visualization',
                    labels={'x': 'PCA1', 'y': 'PCA2', 'z': 'PCA3'},
                    opacity=0.7
                )
                
                fig.show()
            
            return mahalanobis_dist, outliers, chi2_threshold
            
        except np.linalg.LinAlgError:
            print("Covariance matrix is singular. Cannot compute Mahalanobis distance.")
            return None, None, None

# %%
# Apply statistical anomaly detection
stats_viz = StatisticalVisualizations(X_scaled, feature_names, y)

# Z-score analysis
z_scores, anomalies = stats_viz.z_score_analysis()

# IQR analysis
outliers_iqr, bounds = stats_viz.iqr_analysis()

# Mahalanobis distance
mahalanobis_results = stats_viz.mahalanobis_distance()

# %% [markdown]
# ## 5. Clustering-based Anomaly Detection

# %%
class ClusteringVisualizations:
    """Clustering-based anomaly detection visualizations"""
    
    def __init__(self, X, feature_names, y=None):
        self.X = X
        self.feature_names = feature_names
        self.y = y
        
    def dbscan_clustering(self, eps=0.5, min_samples=10):
        """DBSCAN clustering for anomaly detection"""
        
        print("=" * 80)
        print("DBSCAN CLUSTERING ANOMALY DETECTION")
        print("=" * 80)
        
        # Apply DBSCAN
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        clusters = dbscan.fit_predict(self.X)
        
        # Identify anomalies (cluster label = -1)
        anomalies = clusters == -1
        n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
        
        print(f"\n📊 DBSCAN Results:")
        print(f"  Number of clusters: {n_clusters}")
        print(f"  Anomalies detected: {anomalies.sum()} ({anomalies.sum()/len(anomalies)*100:.2f}%)")
        print(f"  Points per cluster:")
        unique, counts = np.unique(clusters, return_counts=True)
        for label, count in zip(unique, counts):
            print(f"    Cluster {label}: {count} points")
        
        # Visualize clusters
        self._visualize_clusters(self.X, clusters, "DBSCAN", eps, min_samples)
        
        # Analyze cluster characteristics
        if n_clusters > 0:
            self._analyze_clusters(self.X, clusters, self.feature_names)
        
        return clusters, anomalies
    
    def kmeans_clustering(self, n_clusters=5):
        """K-means clustering with anomaly detection based on distance to centroids"""
        
        print("\n" + "=" * 80)
        print("K-MEANS CLUSTERING WITH ANOMALY DETECTION")
        print("=" * 80)
        
        # Apply K-means
        kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        cluster_labels = kmeans.fit_predict(self.X)
        centroids = kmeans.cluster_centers_
        
        # Calculate distance to nearest centroid
        distances = np.zeros(len(self.X))
        for i in range(len(self.X)):
            # Distance to assigned centroid
            distances[i] = np.linalg.norm(self.X[i] - centroids[cluster_labels[i]])
        
        # Identify anomalies (points far from their centroids)
        distance_threshold = np.percentile(distances, 95)  # Top 5% as anomalies
        anomalies = distances > distance_threshold
        
        print(f"\n📊 K-means Results:")
        print(f"  Number of clusters: {n_clusters}")
        print(f"  Distance threshold (95th percentile): {distance_threshold:.4f}")
        print(f"  Anomalies detected: {anomalies.sum()} ({anomalies.sum()/len(anomalies)*100:.2f}%)")
        
        # Visualize with distance to centroid
        self._visualize_kmeans(self.X, cluster_labels, centroids, distances, anomalies)
        
        return cluster_labels, centroids, distances, anomalies
    
    def _visualize_clusters(self, X, clusters, method, eps=None, min_samples=None):
        """Visualize clustering results"""
        
        # Reduce to 2D for visualization
        pca = PCA(n_components=2, random_state=42)
        X_2d = pca.fit_transform(X)
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # 1. Cluster visualization
        scatter1 = axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=clusters, 
                                  cmap='tab20', alpha=0.6)
        axes[0].set_xlabel('PCA Component 1')
        axes[0].set_ylabel('PCA Component 2')
        axes[0].set_title(f'{method} Clusters (PCA Projection)')
        plt.colorbar(scatter1, ax=axes[0], label='Cluster Label')
        
        # 2. Anomaly visualization
        anomalies = clusters == -1
        colors = ['blue' if not anom else 'red' for anom in anomalies]
        axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=colors, alpha=0.6)
        axes[1].set_xlabel('PCA Component 1')
        axes[1].set_ylabel('PCA Component 2')
        axes[1].set_title(f'{method} Anomalies (Red = Anomaly)')
        axes[1].legend(handles=[
            mpatches.Patch(color='blue', label='Normal'),
            mpatches.Patch(color='red', label='Anomaly')
        ])
        
        # 3. Compare with fraud labels if available
        if self.y is not None:
            # Color by fraud status
            fraud_colors = ['green' if label == 0 else 'red' for label in self.y]
            axes[2].scatter(X_2d[:, 0], X_2d[:, 1], c=fraud_colors, alpha=0.6)
            axes[2].set_xlabel('PCA Component 1')
            axes[2].set_ylabel('PCA Component 2')
            axes[2].set_title('Actual Fraud Labels (Green = Normal, Red = Fraud)')
            axes[2].legend(handles=[
                mpatches.Patch(color='green', label='Normal'),
                mpatches.Patch(color='red', label='Fraud')
            ])
        
        plt.tight_layout()
        plt.show()
        
        # Interactive visualization
        if eps is not None and min_samples is not None:
            title = f"{method} (eps={eps}, min_samples={min_samples})"
        else:
            title = method
        
        self._interactive_cluster_visualization(X_2d, clusters, title)
    
    def _visualize_kmeans(self, X, cluster_labels, centroids, distances, anomalies):
        """Visualize K-means clustering results"""
        
        # Reduce to 2D for visualization
        pca = PCA(n_components=2, random_state=42)
        X_2d = pca.fit_transform(X)
        centroids_2d = pca.transform(centroids)
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        axes = axes.ravel()
        
        # 1. Cluster visualization
        scatter1 = axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=cluster_labels, 
                                  cmap='tab20', alpha=0.6)
        axes[0].scatter(centroids_2d[:, 0], centroids_2d[:, 1], 
                       c='red', marker='X', s=200, label='Centroids')
        axes[0].set_xlabel('PCA Component 1')
        axes[0].set_ylabel('PCA Component 2')
        axes[0].set_title('K-means Clusters with Centroids')
        axes[0].legend()
        
        # 2. Distance to centroid distribution
        axes[1].hist(distances, bins=50, edgecolor='black', alpha=0.7)
        axes[1].axvline(x=np.percentile(distances, 95), color='red', 
                       linestyle='--', label='95th percentile')
        axes[1].set_xlabel('Distance to Centroid')
        axes[1].set_ylabel('Frequency')
        axes[1].set_title('Distance to Centroid Distribution')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        # 3. Anomaly visualization
        colors = ['blue' if not anom else 'red' for anom in anomalies]
        sizes = [30 if not anom else 100 for anom in anomalies]
        axes[2].scatter(X_2d[:, 0], X_2d[:, 1], c=colors, s=sizes, alpha=0.6)
        axes[2].set_xlabel('PCA Component 1')
        axes[2].set_ylabel('PCA Component 2')
        axes[2].set_title('K-means Anomalies (Size indicates distance)')
        axes[2].legend(handles=[
            mpatches.Patch(color='blue', label='Normal'),
            mpatches.Patch(color='red', label='Anomaly')
        ])
        
        # 4. Silhouette analysis
        from sklearn.metrics import silhouette_samples, silhouette_score
        
        if len(set(cluster_labels)) > 1:
            silhouette_avg = silhouette_score(X, cluster_labels)
            sample_silhouette_values = silhouette_samples(X, cluster_labels)
            
            axes[3].hist(sample_silhouette_values, bins=50, edgecolor='black', alpha=0.7)
            axes[3].axvline(x=silhouette_avg, color='red', linestyle='--',
                           label=f'Avg: {silhouette_avg:.3f}')
            axes[3].set_xlabel('Silhouette Coefficient')
            axes[3].set_ylabel('Frequency')
            axes[3].set_title(f'Silhouette Analysis (Avg: {silhouette_avg:.3f})')
            axes[3].legend()
            axes[3].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Interactive visualization
        self._interactive_kmeans_visualization(X_2d, cluster_labels, distances, anomalies)
    
    def _analyze_clusters(self, X, clusters, feature_names):
        """Analyze cluster characteristics"""
        
        n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
        
        if n_clusters <= 0:
            return
        
        print(f"\n🔍 Cluster Analysis:")
        
        for cluster_id in range(n_clusters):
            cluster_mask = clusters == cluster_id
            cluster_data = X[cluster_mask]
            
            if len(cluster_data) > 0:
                # Calculate mean for each feature
                cluster_means = np.mean(cluster_data, axis=0)
                overall_means = np.mean(X, axis=0)
                
                # Find top differentiating features
                diff = np.abs(cluster_means - overall_means)
                top_features_idx = np.argsort(diff)[-5:][::-1]
                
                print(f"\n  Cluster {cluster_id} ({len(cluster_data)} points):")
                for idx in top_features_idx:
                    feature_name = feature_names[idx]
                    cluster_val = cluster_means[idx]
                    overall_val = overall_means[idx]
                    print(f"    • {feature_name}: {cluster_val:.3f} (overall: {overall_val:.3f})")
    
    def _interactive_cluster_visualization(self, X_2d, clusters, title):
        """Interactive cluster visualization"""
        
        # Create DataFrame for Plotly
        plot_df = pd.DataFrame({
            'x': X_2d[:, 0],
            'y': X_2d[:, 1],
            'cluster': clusters.astype(str),
            'is_anomaly': (clusters == -1).astype(str)
        })
        
        # Add fraud labels if available
        if self.y is not None:
            plot_df['is_fraud'] = self.y.astype(str)
        
        # Create interactive plot
        fig = px.scatter(
            plot_df,
            x='x',
            y='y',
            color='cluster',
            symbol='is_anomaly' if -1 in clusters else None,
            hover_data=['is_anomaly'],
            title=f'{title} - Interactive Visualization',
            labels={'x': 'PCA Component 1', 'y': 'PCA Component 2'},
            opacity=0.7
        )
        
        fig.show()
        
        # 3D visualization if we have more dimensions
        if X_2d.shape[1] == 2 and self.X.shape[1] >= 3:
            pca_3d = PCA(n_components=3, random_state=42)
            X_3d = pca_3d.fit_transform(self.X)
            
            fig_3d = px.scatter_3d(
                x=X_3d[:, 0],
                y=X_3d[:, 1],
                z=X_3d[:, 2],
                color=clusters.astype(str),
                title=f'{title} - 3D Visualization',
                labels={'x': 'PCA1', 'y': 'PCA2', 'z': 'PCA3'},
                opacity=0.7
            )
            
            fig_3d.show()
    
    def _interactive_kmeans_visualization(self, X_2d, cluster_labels, distances, anomalies):
        """Interactive K-means visualization"""
        
        # Create DataFrame for Plotly
        plot_df = pd.DataFrame({
            'x': X_2d[:, 0],
            'y': X_2d[:, 1],
            'cluster': cluster_labels.astype(str),
            'distance': distances,
            'is_anomaly': anomalies.astype(str)
        })
        
        # Interactive scatter plot
        fig = px.scatter(
            plot_df,
            x='x',
            y='y',
            color='cluster',
            size='distance',
            symbol='is_anomaly',
            hover_data=['distance'],
            title='K-means Clustering - Interactive Visualization',
            labels={'x': 'PCA Component 1', 'y': 'PCA Component 2'},
            opacity=0.7
        )
        
        fig.show()

# %%
# Apply clustering-based anomaly detection
clustering_viz = ClusteringVisualizations(X_scaled, feature_names, y)

# DBSCAN clustering
print("\n🔍 Running DBSCAN with different parameters...")

# Try different parameter combinations
param_combinations = [
    (0.3, 10),
    (0.5, 10),
    (0.7, 10),
    (0.5, 5),
    (0.5, 15)
]

dbscan_results = []
for eps, min_samples in param_combinations:
    print(f"\n  Testing eps={eps}, min_samples={min_samples}")
    clusters, anomalies = clustering_viz.dbscan_clustering(eps, min_samples)
    dbscan_results.append({
        'eps': eps,
        'min_samples': min_samples,
        'clusters': clusters,
        'anomalies': anomalies,
        'n_anomalies': anomalies.sum(),
        'anomaly_rate': anomalies.sum() / len(anomalies)
    })

# Compare DBSCAN results
dbscan_df = pd.DataFrame(dbscan_results)
print("\n📊 DBSCAN Parameter Comparison:")
display(dbscan_df[['eps', 'min_samples', 'n_anomalies', 'anomaly_rate']])

# K-means clustering
print("\n🔍 Running K-means with different cluster counts...")
kmeans_results = []

for n_clusters in [3, 5, 7, 10]:
    print(f"\n  Testing n_clusters={n_clusters}")
    cluster_labels, centroids, distances, anomalies = clustering_viz.kmeans_clustering(n_clusters)
    kmeans_results.append({
        'n_clusters': n_clusters,
        'anomalies': anomalies.sum(),
        'anomaly_rate': anomalies.sum() / len(anomalies),
        'avg_distance': distances.mean()
    })

# Compare K-means results
kmeans_df = pd.DataFrame(kmeans_results)
print("\n📊 K-means Parameter Comparison:")
display(kmeans_df)

# %% [markdown]
# ## 6. Machine Learning-based Anomaly Detection

# %%
class MLAnomalyVisualizations:
    """Machine learning-based anomaly detection visualizations"""
    
    def __init__(self, X, feature_names, y=None):
        self.X = X
        self.feature_names = feature_names
        self.y = y
        
    def isolation_forest_analysis(self, contamination='auto'):
        """Isolation Forest analysis and visualization"""
        
        print("=" * 80)
        print("ISOLATION FOREST ANOMALY DETECTION")
        print("=" * 80)
        
        # Train Isolation Forest
        iso_forest = IsolationForest(
            contamination=contamination,
            random_state=42,
            n_estimators=100,
            max_samples='auto'
        )
        
        iso_forest.fit(self.X)
        
        # Get anomaly scores (negative scores are more anomalous)
        anomaly_scores = iso_forest.decision_function(self.X)
        predictions = iso_forest.predict(self.X)
        
        # Convert to binary anomalies
        anomalies = predictions == -1
        
        print(f"\n📊 Isolation Forest Results:")
        print(f"  Contamination parameter: {contamination}")
        print(f"  Anomalies detected: {anomalies.sum()} ({anomalies.sum()/len(anomalies)*100:.2f}%)")
        print(f"  Anomaly score range: [{anomaly_scores.min():.3f}, {anomaly_scores.max():.3f}]")
        
        # Visualizations
        self._visualize_isolation_forest(anomaly_scores, anomalies)
        
        return iso_forest, anomaly_scores, anomalies
    
    def local_outlier_factor_analysis(self, contamination='auto'):
        """Local Outlier Factor analysis and visualization"""
        
        print("\n" + "=" * 80)
        print("LOCAL OUTLIER FACTOR ANOMALY DETECTION")
        print("=" * 80)
        
        # Train LOF
        lof = LocalOutlierFactor(
            contamination=contamination,
            novelty=False,  # We're fitting on all data
            n_neighbors=20
        )
        
        # Get predictions and scores
        predictions = lof.fit_predict(self.X)
        anomaly_scores = -lof.negative_outlier_factor_  # Convert to positive scores
        
        # Convert to binary anomalies
        anomalies = predictions == -1
        
        print(f"\n📊 Local Outlier Factor Results:")
        print(f"  Contamination parameter: {contamination}")
        print(f"  Anomalies detected: {anomalies.sum()} ({anomalies.sum()/len(anomalies)*100:.2f}%)")
        print(f"  Anomaly score range: [{anomaly_scores.min():.3f}, {anomaly_scores.max():.3f}]")
        
        # Visualizations
        self._visualize_lof(anomaly_scores, anomalies)
        
        return lof, anomaly_scores, anomalies
    
    def _visualize_isolation_forest(self, anomaly_scores, anomalies):
        """Visualize Isolation Forest results"""
        
        # Reduce to 2D for visualization
        pca = PCA(n_components=2, random_state=42)
        X_2d = pca.fit_transform(self.X)
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        axes = axes.ravel()
        
        # 1. Anomaly score distribution
        axes[0].hist(anomaly_scores, bins=50, edgecolor='black', alpha=0.7)
        axes[0].axvline(x=0, color='red', linestyle='--', label='Decision boundary')
        axes[0].set_xlabel('Anomaly Score')
        axes[0].set_ylabel('Frequency')
        axes[0].set_title('Isolation Forest Anomaly Score Distribution')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # 2. PCA scatter with anomaly scores
        scatter1 = axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=anomaly_scores, 
                                  cmap='viridis', alpha=0.6)
        axes[1].set_xlabel('PCA Component 1')
        axes[1].set_ylabel('PCA Component 2')
        axes[1].set_title('Isolation Forest Anomaly Scores (PCA)')
        plt.colorbar(scatter1, ax=axes[1], label='Anomaly Score')
        
        # 3. Anomaly detection results
        colors = ['blue' if not anom else 'red' for anom in anomalies]
        axes[2].scatter(X_2d[:, 0], X_2d[:, 1], c=colors, alpha=0.6)
        axes[2].set_xlabel('PCA Component 1')
        axes[2].set_ylabel('PCA Component 2')
        axes[2].set_title('Isolation Forest Anomalies (Red = Anomaly)')
        axes[2].legend(handles=[
            mpatches.Patch(color='blue', label='Normal'),
            mpatches.Patch(color='red', label='Anomaly')
        ])
        
        # 4. Compare with fraud labels if available
        if self.y is not None:
            fraud_anomaly_scores = anomaly_scores[self.y == 1]
            normal_anomaly_scores = anomaly_scores[self.y == 0]
            
            axes[3].hist(normal_anomaly_scores, bins=30, alpha=0.6, 
                        label='Normal', color='blue', density=True)
            axes[3].hist(fraud_anomaly_scores, bins=30, alpha=0.6, 
                        label='Fraud', color='red', density=True)
            axes[3].axvline(x=0, color='green', linestyle='--', label='Decision boundary')
            axes[3].set_xlabel('Anomaly Score')
            axes[3].set_ylabel('Density')
            axes[3].set_title('Anomaly Score Distribution by Fraud Status')
            axes[3].legend()
            axes[3].grid(True, alpha=0.3)
            
            # 5. ROC curve for anomaly scores as fraud predictor
            from sklearn.metrics import roc_curve, auc
            
            fpr, tpr, thresholds = roc_curve(self.y, -anomaly_scores)  # Negative because lower scores are more anomalous
            roc_auc = auc(fpr, tpr)
            
            axes[4].plot(fpr, tpr, color='darkorange', lw=2,
                        label=f'ROC curve (AUC = {roc_auc:.3f})')
            axes[4].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
            axes[4].set_xlabel('False Positive Rate')
            axes[4].set_ylabel('True Positive Rate')
            axes[4].set_title('ROC Curve: Anomaly Score vs Fraud')
            axes[4].legend(loc="lower right")
            axes[4].grid(True, alpha=0.3)
            
            # 6. Precision-Recall curve
            from sklearn.metrics import precision_recall_curve, average_precision_score
            
            precision, recall, _ = precision_recall_curve(self.y, -anomaly_scores)
            avg_precision = average_precision_score(self.y, -anomaly_scores)
            
            axes[5].plot(recall, precision, color='darkgreen', lw=2,
                        label=f'Avg Precision = {avg_precision:.3f}')
            axes[5].set_xlabel('Recall')
            axes[5].set_ylabel('Precision')
            axes[5].set_title('Precision-Recall Curve')
            axes[5].legend(loc="upper right")
            axes[5].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Interactive visualization
        self._interactive_isolation_forest_visualization(X_2d, anomaly_scores, anomalies)
    
    def _visualize_lof(self, anomaly_scores, anomalies):
        """Visualize Local Outlier Factor results"""
        
        # Reduce to 2D for visualization
        pca = PCA(n_components=2, random_state=42)
        X_2d = pca.fit_transform(self.X)
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        axes = axes.ravel()
        
        # 1. Anomaly score distribution
        axes[0].hist(anomaly_scores, bins=50, edgecolor='black', alpha=0.7)
        # Find threshold (95th percentile)
        threshold = np.percentile(anomaly_scores, 95)
        axes[0].axvline(x=threshold, color='red', linestyle='--', 
                       label=f'95th percentile: {threshold:.3f}')
        axes[0].set_xlabel('Anomaly Score (LOF)')
        axes[0].set_ylabel('Frequency')
        axes[0].set_title('Local Outlier Factor Score Distribution')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # 2. PCA scatter with anomaly scores
        scatter1 = axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=anomaly_scores, 
                                  cmap='viridis', alpha=0.6)
        axes[1].set_xlabel('PCA Component 1')
        axes[1].set_ylabel('PCA Component 2')
        axes[1].set_title('LOF Anomaly Scores (PCA)')
        plt.colorbar(scatter1, ax=axes[1], label='Anomaly Score')
        
        # 3. Anomaly detection results
        colors = ['blue' if not anom else 'red' for anom in anomalies]
        sizes = [30 if not anom else 100 for anom in anomalies]
        axes[2].scatter(X_2d[:, 0], X_2d[:, 1], c=colors, s=sizes, alpha=0.6)
        axes[2].set_xlabel('PCA Component 1')
        axes[2].set_ylabel('PCA Component 2')
        axes[2].set_title('LOF Anomalies (Red = Anomaly, Size indicates score)')
        axes[2].legend(handles=[
            mpatches.Patch(color='blue', label='Normal'),
            mpatches.Patch(color='red', label='Anomaly')
        ])
        
        # 4. Local density visualization
        if self.X.shape[0] <= 1000:  # Limit for performance
            from sklearn.neighbors import NearestNeighbors
            
            # Calculate average distance to k nearest neighbors
            n_neighbors = min(20, self.X.shape[0] - 1)
            nbrs = NearestNeighbors(n_neighbors=n_neighbors).fit(self.X)
            distances, _ = nbrs.kneighbors(self.X)
            avg_distances = distances.mean(axis=1)
            
            scatter2 = axes[3].scatter(X_2d[:, 0], X_2d[:, 1], c=avg_distances, 
                                      cmap='plasma', alpha=0.6)
            axes[3].set_xlabel('PCA Component 1')
            axes[3].set_ylabel('PCA Component 2')
            axes[3].set_title(f'Average Distance to {n_neighbors} Nearest Neighbors')
            plt.colorbar(scatter2, ax=axes[3], label='Average Distance')
        
        plt.tight_layout()
        plt.show()
        
        # Interactive visualization
        self._interactive_lof_visualization(X_2d, anomaly_scores, anomalies)
    
    def _interactive_isolation_forest_visualization(self, X_2d, anomaly_scores, anomalies):
        """Interactive Isolation Forest visualization"""
        
        # Create DataFrame for Plotly
        plot_df = pd.DataFrame({
            'x': X_2d[:, 0],
            'y': X_2d[:, 1],
            'anomaly_score': anomaly_scores,
            'is_anomaly': anomalies.astype(str),
            'anomaly_size': np.abs(anomaly_scores) * 10  # Scale for visualization
        })
        
        # Add fraud labels if available
        if self.y is not None:
            plot_df['is_fraud'] = self.y.astype(str)
        
        # Interactive scatter plot
        fig = px.scatter(
            plot_df,
            x='x',
            y='y',
            color='anomaly_score',
            size='anomaly_size',
            symbol='is_anomaly',
            hover_data=['anomaly_score'],
            title='Isolation Forest - Interactive Visualization',
            labels={'x': 'PCA Component 1', 'y': 'PCA Component 2'},
            color_continuous_scale='viridis',
            opacity=0.7
        )
        
        fig.show()
        
        # 3D visualization
        if X_2d.shape[1] == 2 and self.X.shape[1] >= 3:
            pca_3d = PCA(n_components=3, random_state=42)
            X_3d = pca_3d.fit_transform(self.X)
            
            fig_3d = px.scatter_3d(
                x=X_3d[:, 0],
                y=X_3d[:, 1],
                z=X_3d[:, 2],
                color=anomaly_scores,
                size=np.abs(anomaly_scores) * 10,
                title='Isolation Forest - 3D Visualization',
                labels={'x': 'PCA1', 'y': 'PCA2', 'z': 'PCA3'},
                color_continuous_scale='viridis',
                opacity=0.7
            )
            
            fig_3d.show()
    
    def _interactive_lof_visualization(self, X_2d, anomaly_scores, anomalies):
        """Interactive LOF visualization"""
        
        # Create DataFrame for Plotly
        plot_df = pd.DataFrame({
            'x': X_2d[:, 0],
            'y': X_2d[:, 1],
            'anomaly_score': anomaly_scores,
            'is_anomaly': anomalies.astype(str)
        })
        
        # Interactive scatter plot
        fig = px.scatter(
            plot_df,
            x='x',
            y='y',
            color='anomaly_score',
            size='anomaly_score',  # Size proportional to anomaly score
            symbol='is_anomaly',
            hover_data=['anomaly_score'],
            title='Local Outlier Factor - Interactive Visualization',
            labels={'x': 'PCA Component 1', 'y': 'PCA Component 2'},
            color_continuous_scale='plasma',
            opacity=0.7
        )
        
        fig.show()
    
    def compare_ml_methods(self):
        """Compare different ML anomaly detection methods"""
        
        print("=" * 80)
        print("COMPARING ML ANOMALY DETECTION METHODS")
        print("=" * 80)
        
        results = []
        
        # Test different methods
        methods = [
            ('Isolation Forest', self.isolation_forest_analysis),
            ('Local Outlier Factor', self.local_outlier_factor_analysis)
        ]
        
        for method_name, method_func in methods:
            print(f"\n🔍 Testing {method_name}...")
            
            try:
                model, scores, anomalies = method_func(contamination='auto')
                
                # Calculate metrics
                n_anomalies = anomalies.sum()
                anomaly_rate = n_anomalies / len(anomalies)
                score_range = f"[{scores.min():.3f}, {scores.max():.3f}]"
                
                # Compare with fraud labels if available
                if self.y is not None:
                    from sklearn.metrics import roc_auc_score, average_precision_score
                    
                    # For Isolation Forest, lower scores are more anomalous
                    if method_name == 'Isolation Forest':
                        fraud_pred_scores = -scores
                    else:
                        fraud_pred_scores = scores
                    
                    roc_auc = roc_auc_score(self.y, fraud_pred_scores)
                    avg_precision = average_precision_score(self.y, fraud_pred_scores)
                    
                    results.append({
                        'Method': method_name,
                        'Anomalies Detected': n_anomalies,
                        'Anomaly Rate': f"{anomaly_rate*100:.2f}%",
                        'Score Range': score_range,
                        'ROC-AUC': f"{roc_auc:.3f}",
                        'Avg Precision': f"{avg_precision:.3f}"
                    })
                else:
                    results.append({
                        'Method': method_name,
                        'Anomalies Detected': n_anomalies,
                        'Anomaly Rate': f"{anomaly_rate*100:.2f}%",
                        'Score Range': score_range
                    })
                    
            except Exception as e:
                print(f"  Error with {method_name}: {e}")
                results.append({
                    'Method': method_name,
                    'Error': str(e)
                })
        
        # Display comparison
        results_df = pd.DataFrame(results)
        print("\n📊 Comparison Results:")
        display(results_df)
        
        # Visual comparison
        if self.y is not None and 'ROC-AUC' in results_df.columns:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            
            # ROC-AUC comparison
            roc_auc_values = pd.to_numeric(results_df['ROC-AUC'], errors='coerce')
            bars1 = axes[0].bar(range(len(roc_auc_values)), roc_auc_values)
            axes[0].set_xticks(range(len(roc_auc_values)))
            axes[0].set_xticklabels(results_df['Method'], rotation=45)
            axes[0].set_ylabel('ROC-AUC Score')
            axes[0].set_title('ROC-AUC Comparison')
            axes[0].axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Random')
            axes[0].legend()
            
            # Add value labels
            for bar in bars1:
                height = bar.get_height()
                axes[0].text(bar.get_x() + bar.get_width()/2., height,
                           f'{height:.3f}', ha='center', va='bottom')
            
            # Anomaly rate comparison
            anomaly_rates = pd.to_numeric(results_df['Anomaly Rate'].str.replace('%', ''), 
                                        errors='coerce') / 100
            bars2 = axes[1].bar(range(len(anomaly_rates)), anomaly_rates)
            axes[1].set_xticks(range(len(anomaly_rates)))
            axes[1].set_xticklabels(results_df['Method'], rotation=45)
            axes[1].set_ylabel('Anomaly Rate')
            axes[1].set_title('Anomaly Rate Comparison')
            
            # Add value labels
            for bar in bars2:
                height = bar.get_height()
                axes[1].text(bar.get_x() + bar.get_width()/2., height,
                           f'{height:.2%}', ha='center', va='bottom')
            
            plt.tight_layout()
            plt.show()
        
        return results_df

# %%
# Apply ML-based anomaly detection
ml_viz = MLAnomalyVisualizations(X_scaled, feature_names, y)

# Isolation Forest
iso_forest, iso_scores, iso_anomalies = ml_viz.isolation_forest_analysis()

# Local Outlier Factor
lof, lof_scores, lof_anomalies = ml_viz.local_outlier_factor_analysis()

# Compare methods
comparison_df = ml_viz.compare_ml_methods()

# %% [markdown]
# ## 7. Advanced Visualizations

# %%
class AdvancedVisualizations:
    """Advanced anomaly detection visualizations"""
    
    def __init__(self, X, feature_names, y=None):
        self.X = X
        self.feature_names = feature_names
        self.y = y
        
    def radar_chart_anomalies(self, sample_indices=None, n_samples=5):
        """Create radar charts for anomalies"""
        
        print("=" * 80)
        print("RADAR CHART VISUALIZATION OF ANOMALIES")
        print("=" * 80)
        
        if sample_indices is None:
            # Find top anomalies by Isolation Forest score
            iso_forest = IsolationForest(random_state=42)
            iso_forest.fit(self.X)
            anomaly_scores = iso_forest.decision_function(self.X)
            
            # Get top anomalies (most negative scores)
            top_anomaly_indices = np.argsort(anomaly_scores)[:n_samples]
            top_normal_indices = np.argsort(anomaly_scores)[-n_samples:]
            
            sample_indices = list(top_anomaly_indices) + list(top_normal_indices)
            labels = ['Anomaly'] * n_samples + ['Normal'] * n_samples
        
        # Select top features for radar chart
        n_features = min(8, len(self.feature_names))
        
        # Use feature variance to select important features
        feature_variances = np.var(self.X, axis=0)
        top_feature_indices = np.argsort(feature_variances)[-n_features:][::-1]
        top_features = [self.feature_names[i] for i in top_feature_indices]
        
        # Create radar charts
        fig, axes = plt.subplots(2, n_samples, figsize=(4*n_samples, 8))
        
        if n_samples == 1:
            axes = axes.reshape(2, 1)
        
        for col in range(n_samples):
            # Anomaly sample
            anomaly_idx = sample_indices[col]
            anomaly_data = self.X[anomaly_idx, top_feature_indices]
            
            # Normalize for radar chart
            max_vals = np.max(self.X[:, top_feature_indices], axis=0)
            min_vals = np.min(self.X[:, top_feature_indices], axis=0)
            anomaly_normalized = (anomaly_data - min_vals) / (max_vals - min_vals + 1e-8)
            
            # Create radar chart
            self._create_single_radar_chart(anomaly_normalized, top_features, 
                                          axes[0, col], f"Anomaly Sample {col+1}")
            
            # Normal sample
            normal_idx = sample_indices[n_samples + col]
            normal_data = self.X[normal_idx, top_feature_indices]
            normal_normalized = (normal_data - min_vals) / (max_vals - min_vals + 1e-8)
            
            self._create_single_radar_chart(normal_normalized, top_features,
                                          axes[1, col], f"Normal Sample {col+1}")
        
        plt.tight_layout()
        plt.suptitle('Radar Chart Comparison: Anomalies vs Normal Samples', 
                    fontsize=14, y=1.02)
        plt.show()
        
        # Interactive radar chart comparison
        self._interactive_radar_chart(sample_indices, top_feature_indices, top_features, labels)
    
    def _create_single_radar_chart(self, values, features, ax, title):
        """Create a single radar chart"""
        
        # Number of variables
        N = len(features)
        
        # What will be the angle of each axis
        angles = [n / float(N) * 2 * np.pi for n in range(N)]
        angles += angles[:1]  # Close the loop
        
        # Initialise the spider plot
        ax = plt.subplot(111, polar=True)
        
        # Draw one axe per variable + add labels
        plt.xticks(angles[:-1], features, size=8)
        
        # Draw ylabels
        ax.set_rlabel_position(0)
        plt.yticks([0.25, 0.5, 0.75], ["0.25", "0.5", "0.75"], color="grey", size=7)
        plt.ylim(0, 1)
        
        # Plot data
        values = list(values)
        values += values[:1]  # Close the loop
        ax.plot(angles, values, linewidth=1, linestyle='solid')
        ax.fill(angles, values, 'b', alpha=0.1)
        
        # Add a title
        plt.title(title, size=11, y=1.1)
    
    def _interactive_radar_chart(self, sample_indices, feature_indices, features, labels):
        """Interactive radar chart comparison"""
        
        import plotly.graph_objects as go
        
        fig = go.Figure()
        
        for i, idx in enumerate(sample_indices):
            sample_data = self.X[idx, feature_indices]
            
            # Normalize
            max_vals = np.max(self.X[:, feature_indices], axis=0)
            min_vals = np.min(self.X[:, feature_indices], axis=0)
            normalized_data = (sample_data - min_vals) / (max_vals - min_vals + 1e-8)
            
            fig.add_trace(go.Scatterpolar(
                r=np.concatenate([normalized_data, [normalized_data[0]]]),  # Close the loop
                theta=features + [features[0]],  # Close the loop
                fill='toself',
                name=f"{labels[i]} {i+1}"
            ))
        
        fig.update_layout(
            polar=dict(
                radialaxis=dict(
                    visible=True,
                    range=[0, 1]
                )
            ),
            showlegend=True,
            title='Interactive Radar Chart Comparison',
            height=600
        )
        
        fig.show()
    
    def parallel_coordinates(self, n_samples=100):
        """Parallel coordinates plot for anomaly visualization"""
        
        print("\n" + "=" * 80)
        print("PARALLEL COORDINATES VISUALIZATION")
        print("=" * 80)
        
        # Sample data
        if len(self.X) > n_samples:
            sample_indices = np.random.choice(len(self.X), n_samples, replace=False)
            X_sample = self.X[sample_indices]
            y_sample = self.y[sample_indices] if self.y is not None else None
        else:
            X_sample = self.X
            y_sample = self.y
        
        # Get anomaly scores
        iso_forest = IsolationForest(random_state=42)
        iso_forest.fit(self.X)
        anomaly_scores = iso_forest.decision_function(X_sample)
        
        # Select top features
        n_features = min(10, len(self.feature_names))
        feature_variances = np.var(self.X, axis=0)
        top_feature_indices = np.argsort(feature_variances)[-n_features:][::-1]
        top_features = [self.feature_names[i] for i in top_feature_indices]
        
        # Create DataFrame for parallel coordinates
        plot_data = pd.DataFrame(X_sample[:, top_feature_indices], columns=top_features)
        plot_data['anomaly_score'] = anomaly_scores
        plot_data['is_anomaly'] = (anomaly_scores < 0).astype(str)
        
        if y_sample is not None:
            plot_data['is_fraud'] = y_sample.astype(str)
        
        # Normalize features for better visualization
        for feature in top_features:
            plot_data[feature] = (plot_data[feature] - plot_data[feature].min()) / \
                                (plot_data[feature].max() - plot_data[feature].min() + 1e-8)
        
        # Create parallel coordinates plot
        fig = px.parallel_coordinates(
            plot_data,
            dimensions=top_features + ['anomaly_score'],
            color='anomaly_score',
            color_continuous_scale=px.colors.diverging.RdYlBu_r,
            labels={col: col.replace('_', ' ').title() for col in plot_data.columns},
            title='Parallel Coordinates Plot with Anomaly Scores'
        )
        
        fig.show()
        
        # Alternative: Color by anomaly status
        if 'is_anomaly' in plot_data.columns:
            fig2 = px.parallel_coordinates(
                plot_data,
                dimensions=top_features,
                color='is_anomaly',
                color_discrete_map={'True': 'red', 'False': 'blue'},
                labels={col: col.replace('_', ' ').title() for col in top_features},
                title='Parallel Coordinates Plot Colored by Anomaly Status'
            )
            
            fig2.show()
    
    def heatmap_analysis(self):
        """Heatmap analysis of anomalies"""
        
        print("\n" + "=" * 80)
        print("HEATMAP ANALYSIS OF ANOMALIES")
        print("=" * 80)
        
        # Get anomaly scores
        iso_forest = IsolationForest(random_state=42)
        iso_forest.fit(self.X)
        anomaly_scores = iso_forest.decision_function(self.X)
        
        # Identify anomalies
        anomalies = anomaly_scores < 0
        
        # Select subset for heatmap
        n_samples = min(50, len(self.X))
        n_features = min(20, len(self.feature_names))
        
        # Get top anomalous samples
        anomaly_indices = np.argsort(anomaly_scores)[:n_samples]
        
        # Get top features by variance
        feature_variances = np.var(self.X, axis=0)
        top_feature_indices = np.argsort(feature_variances)[-n_features:][::-1]
        top_features = [self.feature_names[i] for i in top_feature_indices]
        
        # Create heatmap data
        heatmap_data = self.X[anomaly_indices][:, top_feature_indices]
        
        # Normalize for better visualization
        heatmap_data_normalized = (heatmap_data - heatmap_data.min(axis=0)) / \
                                 (heatmap_data.max(axis=0) - heatmap_data.min(axis=0) + 1e-8)
        
        # Create heatmap
        fig, axes = plt.subplots(1, 2, figsize=(15, 8))
        
        # Heatmap of feature values
        im1 = axes[0].imshow(heatmap_data_normalized.T, aspect='auto', 
                            cmap='viridis', interpolation='nearest')
        axes[0].set_xlabel('Sample Index (sorted by anomaly score)')
        axes[0].set_ylabel('Feature')
        axes[0].set_title('Feature Values for Top Anomalies')
        axes[0].set_yticks(range(len(top_features)))
        axes[0].set_yticklabels(top_features, fontsize=8)
        plt.colorbar(im1, ax=axes[0], label='Normalized Value')
        
        # Correlation heatmap of anomalies
        anomaly_correlation = np.corrcoef(heatmap_data.T)
        im2 = axes[1].imshow(anomaly_correlation, aspect='auto', 
                            cmap='coolwarm', vmin=-1, vmax=1)
        axes[1].set_xlabel('Feature')
        axes[1].set_ylabel('Feature')
        axes[1].set_title('Feature Correlation in Anomalies')
        axes[1].set_xticks(range(len(top_features)))
        axes[1].set_yticks(range(len(top_features)))
        axes[1].set_xticklabels(top_features, rotation=90, fontsize=8)
        axes[1].set_yticklabels(top_features, fontsize=8)
        plt.colorbar(im2, ax=axes[1], label='Correlation')
        
        # Add correlation values
        for i in range(len(top_features)):
            for j in range(len(top_features)):
                text = axes[1].text(j, i, f'{anomaly_correlation[i, j]:.2f}',
                                   ha="center", va="center", 
                                   color="white" if abs(anomaly_correlation[i, j]) > 0.5 else "black",
                                   fontsize=6)
        
        plt.tight_layout()
        plt.show()
        
        # Interactive heatmap
        self._interactive_heatmap(heatmap_data_normalized, top_features, anomaly_scores[anomaly_indices])
    
    def _interactive_heatmap(self, heatmap_data, features, anomaly_scores):
        """Interactive heatmap visualization"""
        
        # Create DataFrame for Plotly
        heatmap_df = pd.DataFrame(heatmap_data, columns=features)
        heatmap_df['anomaly_score'] = anomaly_scores
        heatmap_df['sample_index'] = range(len(heatmap_df))
        
        # Melt for heatmap
        heatmap_melted = heatmap_df.melt(id_vars=['sample_index', 'anomaly_score'], 
                                        value_vars=features,
                                        var_name='feature', value_name='value')
        
        # Create interactive heatmap
        fig = px.density_heatmap(
            heatmap_melted,
            x='sample_index',
            y='feature',
            z='value',
            histfunc='avg',
            title='Interactive Heatmap of Anomalous Samples',
            labels={'sample_index': 'Sample Index', 'feature': 'Feature', 'value': 'Normalized Value'},
            color_continuous_scale='viridis'
        )
        
        fig.update_layout(
            height=600,
            xaxis_title='Sample Index (sorted by anomaly severity)',
            yaxis_title='Feature'
        )
        
        fig.show()
    
    def anomaly_timeline(self, timestamps=None):
        """Timeline visualization of anomalies"""
        
        print("\n" + "=" * 80)
        print("ANOMALY TIMELINE VISUALIZATION")
        print("=" * 80)
        
        # Get anomaly scores
        iso_forest = IsolationForest(random_state=42)
        iso_forest.fit(self.X)
        anomaly_scores = iso_forest.decision_function(self.X)
        
        # Create timeline data
        if timestamps is None:
            # Create synthetic timestamps if not provided
            timestamps = pd.date_range(start='2024-01-01', periods=len(self.X), freq='H')
        
        timeline_df = pd.DataFrame({
            'timestamp': timestamps[:len(anomaly_scores)],
            'anomaly_score': anomaly_scores,
            'is_anomaly': (anomaly_scores < 0).astype(int),
            'anomaly_magnitude': np.abs(anomaly_scores)
        })
        
        if self.y is not None:
            timeline_df['is_fraud'] = self.y
        
        # Resample for smoother visualization
        timeline_df.set_index('timestamp', inplace=True)
        resampled = timeline_df.resample('6H').mean()
        
        fig, axes = plt.subplots(3, 1, figsize=(15, 12))
        
        # 1. Anomaly score over time
        axes[0].plot(resampled.index, resampled['anomaly_score'], alpha=0.7)
        axes[0].fill_between(resampled.index, 0, resampled['anomaly_score'], 
                            where=resampled['anomaly_score'] < 0, 
                            color='red', alpha=0.3, label='Anomaly Region')
        axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Anomaly Threshold')
        axes[0].set_xlabel('Time')
        axes[0].set_ylabel('Anomaly Score')
        axes[0].set_title('Anomaly Score Timeline')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # 2. Anomaly density over time
        axes[1].bar(resampled.index, resampled['is_anomaly'], width=0.1, alpha=0.7)
        axes[1].set_xlabel('Time')
        axes[1].set_ylabel('Anomaly Density')
        axes[1].set_title('Anomaly Density Timeline')
        axes[1].grid(True, alpha=0.3)
        
        # 3. Compare with fraud if available
        if 'is_fraud' in resampled.columns:
            axes[2].plot(resampled.index, resampled['is_fraud'], alpha=0.7, label='Fraud Rate')
            axes[2].plot(resampled.index, resampled['is_anomaly'], alpha=0.7, label='Anomaly Rate')
            axes[2].set_xlabel('Time')
            axes[2].set_ylabel('Rate')
            axes[2].set_title('Fraud vs Anomaly Rate Comparison')
            axes[2].legend()
            axes[2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Interactive timeline
        self._interactive_timeline(timeline_df)
    
    def _interactive_timeline(self, timeline_df):
        """Interactive timeline visualization"""
        
        fig = go.Figure()
        
        # Add anomaly score line
        fig.add_trace(go.Scatter(
            x=timeline_df.index,
            y=timeline_df['anomaly_score'],
            mode='lines',
            name='Anomaly Score',
            line=dict(color='blue', width=1),
            opacity=0.7
        ))
        
        # Add anomaly regions
        fig.add_trace(go.Scatter(
            x=timeline_df.index,
            y=np.where(timeline_df['anomaly_score'] < 0, timeline_df['anomaly_score'], np.nan),
            mode='lines',
            name='Anomaly Regions',
            line=dict(color='red', width=2),
            fill='tozeroy',
            fillcolor='rgba(255, 0, 0, 0.2)'
        ))
        
        # Add fraud markers if available
        if 'is_fraud' in timeline_df.columns:
            fraud_indices = timeline_df[timeline_df['is_fraud'] > 0].index
            fraud_scores = timeline_df.loc[fraud_indices, 'anomaly_score']
            
            fig.add_trace(go.Scatter(
                x=fraud_indices,
                y=fraud_scores,
                mode='markers',
                name='Fraud Cases',
                marker=dict(
                    color='red',
                    size=10,
                    symbol='x'
                )
            ))
        
        fig.update_layout(
            title='Interactive Anomaly Timeline',
            xaxis_title='Time',
            yaxis_title='Anomaly Score',
            hovermode='x unified',
            height=500
        )
        
        fig.show()

# %%
# Apply advanced visualizations
advanced_viz = AdvancedVisualizations(X_scaled, feature_names, y)

# Radar charts
advanced_viz.radar_chart_anomalies(n_samples=3)

# Parallel coordinates
advanced_viz.parallel_coordinates(n_samples=100)

# Heatmap analysis
advanced_viz.heatmap_analysis()

# Anomaly timeline
# Create synthetic timestamps if not available
if 'timestamp' in df.columns:
    timestamps = pd.to_datetime(df['timestamp'])
else:
    timestamps = pd.date_range(start='2024-01-01', periods=len(X_scaled), freq='H')

advanced_viz.anomaly_timeline(timestamps=timestamps[:len(X_scaled)])

# %% [markdown]
# ## 8. Anomaly Detection Dashboard

# %%
class AnomalyDashboard:
    """Comprehensive anomaly detection dashboard"""
    
    def __init__(self, X, feature_names, y=None):
        self.X = X
        self.feature_names = feature_names
        self.y = y
        self.results = {}
        
    def run_all_detectors(self):
        """Run all anomaly detectors"""
        
        print("=" * 80)
        print("RUNNING ALL ANOMALY DETECTORS")
        print("=" * 80)
        
        detectors = {
            'z_score': self._run_z_score,
            'iqr': self._run_iqr,
            'isolation_forest': self._run_isolation_forest,
            'local_outlier_factor': self._run_lof,
            'dbscan': self._run_dbscan,
            'kmeans': self._run_kmeans
        }
        
        for name, detector_func in detectors.items():
            print(f"\n🔍 Running {name.replace('_', ' ').title()}...")
            try:
                result = detector_func()
                self.results[name] = result
                print(f"  ✅ Completed - Anomalies: {result['n_anomalies']} ({result['anomaly_rate']*100:.2f}%)")
            except Exception as e:
                print(f"  ❌ Failed: {e}")
                self.results[name] = {'error': str(e)}
        
        print(f"\n🎯 All detectors completed: {len(self.results)}/{len(detectors)} successful")
        
        return self.results
    
    def _run_z_score(self):
        """Run Z-score detector"""
        z_scores = np.abs(stats.zscore(self.X, axis=0))
        anomalies = (z_scores > 3).any(axis=1)
        
        return {
            'anomalies': anomalies,
            'n_anomalies': anomalies.sum(),
            'anomaly_rate': anomalies.sum() / len(anomalies),
            'scores': z_scores.max(axis=1)
        }
    
    def _run_iqr(self):
        """Run IQR detector"""
        Q1 = np.percentile(self.X, 25, axis=0)
        Q3 = np.percentile(self.X, 75, axis=0)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        anomalies = ((self.X < lower_bound) | (self.X > upper_bound)).any(axis=1)
        
        return {
            'anomalies': anomalies,
            'n_anomalies': anomalies.sum(),
            'anomaly_rate': anomalies.sum() / len(anomalies),
            'bounds': (lower_bound, upper_bound)
        }
    
    def _run_isolation_forest(self):
        """Run Isolation Forest"""
        iso_forest = IsolationForest(random_state=42, contamination='auto')
        predictions = iso_forest.fit_predict(self.X)
        anomalies = predictions == -1
        
        return {
            'anomalies': anomalies,
            'n_anomalies': anomalies.sum(),
            'anomaly_rate': anomalies.sum() / len(anomalies),
            'model': iso_forest
        }
    
    def _run_lof(self):
        """Run Local Outlier Factor"""
        lof = LocalOutlierFactor(contamination='auto', novelty=False)
        predictions = lof.fit_predict(self.X)
        anomalies = predictions == -1
        
        return {
            'anomalies': anomalies,
            'n_anomalies': anomalies.sum(),
            'anomaly_rate': anomalies.sum() / len(anomalies),
            'model': lof
        }
    
    def _run_dbscan(self):
        """Run DBSCAN"""
        dbscan = DBSCAN(eps=0.5, min_samples=10)
        clusters = dbscan.fit_predict(self.X)
        anomalies = clusters == -1
        
        return {
            'anomalies': anomalies,
            'n_anomalies': anomalies.sum(),
            'anomaly_rate': anomalies.sum() / len(anomalies),
            'clusters': clusters,
            'model': dbscan
        }
    
    def _run_kmeans(self):
        """Run K-means with distance threshold"""
        from sklearn.cluster import KMeans
        
        kmeans = KMeans(n_clusters=5, random_state=42)
        clusters = kmeans.fit_predict(self.X)
        centroids = kmeans.cluster_centers_
        
        # Calculate distance to centroid
        distances = np.zeros(len(self.X))
        for i in range(len(self.X)):
            distances[i] = np.linalg.norm(self.X[i] - centroids[clusters[i]])
        
        # Top 5% as anomalies
        threshold = np.percentile(distances, 95)
        anomalies = distances > threshold
        
        return {
            'anomalies': anomalies,
            'n_anomalies': anomalies.sum(),
            'anomaly_rate': anomalies.sum() / len(anomalies),
            'distances': distances,
            'threshold': threshold,
            'model': kmeans
        }
    
    def create_dashboard(self):
        """Create comprehensive dashboard"""
        
        if not self.results:
            self.run_all_detectors()
        
        print("\n" + "=" * 80)
        print("ANOMALY DETECTION DASHBOARD")
        print("=" * 80)
        
        # Summary statistics
        summary_data = []
        for name, result in self.results.items():
            if 'error' not in result:
                summary_data.append({
                    'Method': name.replace('_', ' ').title(),
                    'Anomalies': result['n_anomalies'],
                    'Anomaly Rate': f"{result['anomaly_rate']*100:.2f}%",
                    'Success Rate': '✅' if result['n_anomalies'] > 0 else '⚠️'
                })
        
        summary_df = pd.DataFrame(summary_data)
        print("\n📊 Summary Statistics:")
        display(summary_df)
        
        # Visualizations
        self._create_comparison_visualizations()
        
        # Agreement analysis
        self._analyze_agreement()
        
        # Feature importance for anomalies
        self._analyze_anomaly_features()
        
        return summary_df
    
    def _create_comparison_visualizations(self):
        """Create comparison visualizations"""
        
        # Prepare data for comparison
        method_names = []
        anomaly_rates = []
        anomaly_counts = []
        
        for name, result in self.results.items():
            if 'error' not in result:
                method_names.append(name.replace('_', ' ').title())
                anomaly_rates.append(result['anomaly_rate'] * 100)
                anomaly_counts.append(result['n_anomalies'])
        
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.ravel()
        
        # 1. Anomaly rate comparison
        bars1 = axes[0].bar(method_names, anomaly_rates)
        axes[0].set_xlabel('Method')
        axes[0].set_ylabel('Anomaly Rate (%)')
        axes[0].set_title('Anomaly Rate by Method')
        axes[0].tick_params(axis='x', rotation=45)
        
        for bar in bars1:
            height = bar.get_height()
            axes[0].text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.1f}%', ha='center', va='bottom')
        
        # 2. Anomaly count comparison
        bars2 = axes[1].bar(method_names, anomaly_counts)
        axes[1].set_xlabel('Method')
        axes[1].set_ylabel('Number of Anomalies')
        axes[1].set_title('Anomaly Count by Method')
        axes[1].tick_params(axis='x', rotation=45)
        
        for bar in bars2:
            height = bar.get_height()
            axes[1].text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:,}', ha='center', va='bottom')
        
        # 3. Method agreement matrix
        if len(method_names) >= 2:
            agreement_matrix = np.zeros((len(method_names), len(method_names)))
            
            for i, (name1, result1) in enumerate(self.results.items()):
                for j, (name2, result2) in enumerate(self.results.items()):
                    if 'error' not in result1 and 'error' not in result2:
                        agreement = np.mean(result1['anomalies'] == result2['anomalies'])
                        agreement_matrix[i, j] = agreement
            
            im = axes[2].imshow(agreement_matrix, cmap='YlOrRd', vmin=0, vmax=1)
            axes[2].set_xticks(range(len(method_names)))
            axes[2].set_yticks(range(len(method_names)))
            axes[2].set_xticklabels([m[:10] for m in method_names], rotation=45)
            axes[2].set_yticklabels([m[:10] for m in method_names])
            axes[2].set_title('Method Agreement Matrix')
            plt.colorbar(im, ax=axes[2], label='Agreement Score')
            
            # Add agreement values
            for i in range(len(method_names)):
                for j in range(len(method_names)):
                    axes[2].text(j, i, f'{agreement_matrix[i, j]:.2f}',
                               ha='center', va='center', 
                               color='white' if agreement_matrix[i, j] > 0.5 else 'black')
        
        # 4. PCA visualization with combined anomalies
        pca = PCA(n_components=2, random_state=42)
        X_2d = pca.fit_transform(self.X)
        
        # Combine anomalies from all methods
        combined_anomalies = np.zeros(len(self.X), dtype=bool)
        for result in self.results.values():
            if 'error' not in result:
                combined_anomalies = combined_anomalies | result['anomalies']
        
        axes[3].scatter(X_2d[:, 0], X_2d[:, 1], c='blue', alpha=0.3, label='Normal')
        axes[3].scatter(X_2d[combined_anomalies, 0], X_2d[combined_anomalies, 1], 
                       c='red', alpha=0.6, label='Anomaly (any method)')
        axes[3].set_xlabel('PCA Component 1')
        axes[3].set_ylabel('PCA Component 2')
        axes[3].set_title('Combined Anomaly Detection')
        axes[3].legend()
        
        # 5. Compare with fraud labels if available
        if self.y is not None:
            # Calculate precision and recall for each method
            precision_scores = []
            recall_scores = []
            
            for name, result in self.results.items():
                if 'error' not in result:
                    from sklearn.metrics import precision_score, recall_score
                    precision = precision_score(self.y, result['anomalies'], zero_division=0)
                    recall = recall_score(self.y, result['anomalies'], zero_division=0)
                    precision_scores.append(precision)
                    recall_scores.append(recall)
                else:
                    precision_scores.append(0)
                    recall_scores.append(0)
            
            x = np.arange(len(method_names))
            width = 0.35
            
            axes[4].bar(x - width/2, precision_scores, width, label='Precision')
            axes[4].bar(x + width/2, recall_scores, width, label='Recall')
            axes[4].set_xlabel('Method')
            axes[4].set_ylabel('Score')
            axes[4].set_title('Precision and Recall vs Fraud Labels')
            axes[4].set_xticks(x)
            axes[4].set_xticklabels([m[:10] for m in method_names], rotation=45)
            axes[4].legend()
            axes[4].set_ylim([0, 1])
            
            # Add value labels
            for i, (prec, rec) in enumerate(zip(precision_scores, recall_scores)):
                axes[4].text(i - width/2, prec + 0.01, f'{prec:.2f}', 
                           ha='center', va='bottom', fontsize=8)
                axes[4].text(i + width/2, rec + 0.01, f'{rec:.2f}', 
                           ha='center', va='bottom', fontsize=8)
        
        # 6. ROC curve for combined detection
        if self.y is not None and len(method_names) >= 2:
            # Use Isolation Forest scores as example
            if 'isolation_forest' in self.results and 'error' not in self.results['isolation_forest']:
                iso_result = self.results['isolation_forest']
                if hasattr(iso_result['model'], 'decision_function'):
                    scores = iso_result['model'].decision_function(self.X)
                    
                    from sklearn.metrics import roc_curve, auc
                    fpr, tpr, _ = roc_curve(self.y, -scores)  # Negative for anomaly scores
                    roc_auc = auc(fpr, tpr)
                    
                    axes[5].plot(fpr, tpr, color='darkorange', lw=2,
                                label=f'ROC curve (AUC = {roc_auc:.3f})')
                    axes[5].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
                    axes[5].set_xlabel('False Positive Rate')
                    axes[5].set_ylabel('True Positive Rate')
                    axes[5].set_title('ROC Curve: Isolation Forest')
                    axes[5].legend(loc="lower right")
                    axes[5].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Interactive dashboard
        self._create_interactive_dashboard(method_names, anomaly_rates, anomaly_counts)
    
    def _create_interactive_dashboard(self, method_names, anomaly_rates, anomaly_counts):
        """Create interactive dashboard"""
        
        # Create interactive comparison
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('Anomaly Rate by Method', 'Anomaly Count by Method',
                          'Method Comparison', 'Performance Metrics'),
            specs=[[{'type': 'bar'}, {'type': 'bar'}],
                   [{'type': 'scatter'}, {'type': 'bar'}]]
        )
        
        # 1. Anomaly rate bar chart
        fig.add_trace(
            go.Bar(x=method_names, y=anomaly_rates, name='Anomaly Rate'),
            row=1, col=1
        )
        
        # 2. Anomaly count bar chart
        fig.add_trace(
            go.Bar(x=method_names, y=anomaly_counts, name='Anomaly Count'),
            row=1, col=2
        )
        
        # 3. Scatter plot comparison
        if len(method_names) >= 2:
            # Use first two methods for scatter
            method1 = list(self.results.keys())[0]
            method2 = list(self.results.keys())[1]
            
            if 'error' not in self.results[method1] and 'error' not in self.results[method2]:
                pca = PCA(n_components=2, random_state=42)
                X_2d = pca.fit_transform(self.X)
                
                anomalies1 = self.results[method1]['anomalies']
                anomalies2 = self.results[method2]['anomalies']
                
                # Create categories
                categories = []
                for a1, a2 in zip(anomalies1, anomalies2):
                    if a1 and a2:
                        categories.append('Both')
                    elif a1:
                        categories.append(method1.replace('_', ' ').title())
                    elif a2:
                        categories.append(method2.replace('_', ' ').title())
                    else:
                        categories.append('None')
                
                fig.add_trace(
                    go.Scatter(
                        x=X_2d[:, 0],
                        y=X_2d[:, 1],
                        mode='markers',
                        marker=dict(
                            color=categories,
                            colorscale='viridis'
                        ),
                        text=categories,
                        name='Method Comparison'
                    ),
                    row=2, col=1
                )
        
        # 4. Performance metrics if fraud labels available
        if self.y is not None:
            precision_scores = []
            recall_scores = []
            
            for name, result in self.results.items():
                if 'error' not in result:
                    from sklearn.metrics import precision_score, recall_score
                    precision = precision_score(self.y, result['anomalies'], zero_division=0)
                    recall = recall_score(self.y, result['anomalies'], zero_division=0)
                    precision_scores.append(precision)
                    recall_scores.append(recall)
            
            fig.add_trace(
                go.Bar(x=method_names, y=precision_scores, name='Precision'),
                row=2, col=2
            )
            
            fig.add_trace(
                go.Bar(x=method_names, y=recall_scores, name='Recall'),
                row=2, col=2
            )
        
        fig.update_layout(
            height=800,
            showlegend=True,
            title_text="Anomaly Detection Dashboard"
        )
        
        fig.show()
    
    def _analyze_agreement(self):
        """Analyze agreement between methods"""
        
        print("\n🤝 Method Agreement Analysis:")
        
        # Get successful methods
        successful_methods = []
        successful_results = []
        
        for name, result in self.results.items():
            if 'error' not in result:
                successful_methods.append(name)
                successful_results.append(result['anomalies'])
        
        if len(successful_methods) < 2:
            print("  Need at least 2 successful methods for agreement analysis")
            return
        
        # Calculate pairwise agreement
        agreement_matrix = np.zeros((len(successful_methods), len(successful_methods)))
        
        for i in range(len(successful_methods)):
            for j in range(len(successful_methods)):
                agreement = np.mean(successful_results[i] == successful_results[j])
                agreement_matrix[i, j] = agreement
        
        print(f"\n  Pairwise Agreement Matrix:")
        agreement_df = pd.DataFrame(
            agreement_matrix,
            index=[m.replace('_', ' ').title() for m in successful_methods],
            columns=[m.replace('_', ' ').title() for m in successful_methods]
        )
        display(agreement_df)
        
        # Find consensus anomalies
        consensus_threshold = 0.5  # Majority vote
        consensus_anomalies = np.mean(successful_results, axis=0) > consensus_threshold
        
        print(f"\n  Consensus Analysis (threshold: {consensus_threshold*100:.0f}% agreement):")
        print(f"    Consensus anomalies: {consensus_anomalies.sum()} ({consensus_anomalies.sum()/len(consensus_anomalies)*100:.2f}%)")
        
        # Compare with individual methods
        print(f"\n  Consensus vs Individual Methods:")
        for i, method in enumerate(successful_methods):
            agreement = np.mean(consensus_anomalies == successful_results[i])
            print(f"    {method.replace('_', ' ').title()}: {agreement:.3f}")
        
        # Compare with fraud labels if available
        if self.y is not None:
            from sklearn.metrics import precision_score, recall_score, f1_score
            
            consensus_precision = precision_score(self.y, consensus_anomalies, zero_division=0)
            consensus_recall = recall_score(self.y, consensus_anomalies, zero_division=0)
            consensus_f1 = f1_score(self.y, consensus_anomalies, zero_division=0)
            
            print(f"\n  Consensus Performance vs Fraud Labels:")
            print(f"    Precision: {consensus_precision:.3f}")
            print(f"    Recall: {consensus_recall:.3f}")
            print(f"    F1-Score: {consensus_f1:.3f}")
    
    def _analyze_anomaly_features(self):
        """Analyze which features are most indicative of anomalies"""
        
        print("\n🔍 Feature Analysis for Anomalies:")
        
        # Use Isolation Forest feature importance if available
        if 'isolation_forest' in self.results and 'error' not in self.results['isolation_forest']:
            iso_result = self.results['isolation_forest']
            
            if hasattr(iso_result['model'], 'feature_importances_'):
                importances = iso_result['model'].feature_importances_
                
                importance_df = pd.DataFrame({
                    'feature': self.feature_names,
                    'importance': importances
                }).sort_values('importance', ascending=False)
                
                print(f"\n  Isolation Forest Feature Importance (Top 10):")
                display(importance_df.head(10))
                
                # Visualize feature importance
                fig, axes = plt.subplots(1, 2, figsize=(15, 6))
                
                # Bar chart
                top_features = importance_df.head(10)
                axes[0].barh(range(len(top_features)), top_features['importance'])
                axes[0].set_yticks(range(len(top_features)))
                axes[0].set_yticklabels(top_features['feature'])
                axes[0].set_xlabel('Importance')
                axes[0].set_title('Top 10 Features for Anomaly Detection')
                axes[0].invert_yaxis()
                
                # Cumulative importance
                cumulative_importance = np.cumsum(importance_df['importance'])
                cumulative_pct = cumulative_importance / cumulative_importance[-1] * 100
                
                axes[1].plot(range(1, len(cumulative_pct) + 1), cumulative_pct, marker='o')
                axes[1].axhline(y=80, color='r', linestyle='--', alpha=0.5, label='80% threshold')
                axes[1].axhline(y=90, color='g', linestyle='--', alpha=0.5, label='90% threshold')
                axes[1].set_xlabel('Number of Features')
                axes[1].set_ylabel('Cumulative Importance (%)')
                axes[1].set_title('Cumulative Feature Importance')
                axes[1].legend()
                axes[1].grid(True, alpha=0.3)
                
                # Find number of features for 80% and 90% importance
                n_features_80 = np.argmax(cumulative_pct >= 80) + 1
                n_features_90 = np.argmax(cumulative_pct >= 90) + 1
                
                print(f"  Features needed for 80% importance: {n_features_80}")
                print(f"  Features needed for 90% importance: {n_features_90}")
                
                plt.tight_layout()
                plt.show()
        
        # Statistical analysis of anomalies
        print("\n  Statistical Analysis of Anomalous vs Normal Points:")
        
        # Use consensus anomalies if available, otherwise use Isolation Forest
        if 'consensus_anomalies' in locals():
            anomalies = consensus_anomalies
        elif 'isolation_forest' in self.results and 'error' not in self.results['isolation_forest']:
            anomalies = self.results['isolation_forest']['anomalies']
        else:
            print("  No anomaly labels available for analysis")
            return
        
        # Calculate statistics for anomalous vs normal points
        normal_data = self.X[~anomalies]
        anomaly_data = self.X[anomalies]
        
        if len(anomaly_data) > 0:
            # Calculate mean difference
            normal_mean = np.mean(normal_data, axis=0)
            anomaly_mean = np.mean(anomaly_data, axis=0)
            mean_diff = np.abs(anomaly_mean - normal_mean)
            
            # Calculate effect size (Cohen's d)
            normal_std = np.std(normal_data, axis=0)
            anomaly_std = np.std(anomaly_data, axis=0)
            pooled_std = np.sqrt((normal_std**2 + anomaly_std**2) / 2)
            cohens_d = mean_diff / (pooled_std + 1e-8)
            
            # Create analysis DataFrame
            analysis_df = pd.DataFrame({
                'feature': self.feature_names,
                'normal_mean': normal_mean,
                'anomaly_mean': anomaly_mean,
                'mean_diff': mean_diff,
                'cohens_d': cohens_d
            }).sort_values('cohens_d', ascending=False)
            
            print(f"\n  Top 10 Features Differentiating Anomalies (by Cohen's d):")
            display(analysis_df.head(10))
            
            # Visualize top differentiating features
            top_n = min(5, len(analysis_df))
            top_features = analysis_df.head(top_n)
            
            fig, axes = plt.subplots(1, top_n, figsize=(4*top_n, 6))
            
            if top_n == 1:
                axes = [axes]
            
            for idx, (_, row) in enumerate(top_features.iterrows()):
                feature_idx = self.feature_names.index(row['feature'])
                
                # Create violin plot
                data_to_plot = [normal_data[:, feature_idx], anomaly_data[:, feature_idx]]
                
                axes[idx].violinplot(data_to_plot, showmeans=True, showmedians=True)
                axes[idx].set_xticks([1, 2])
                axes[idx].set_xticklabels(['Normal', 'Anomaly'])
                axes[idx].set_ylabel('Value')
                axes[idx].set_title(f"{row['feature']}\nCohen's d = {row['cohens_d']:.2f}")
                axes[idx].grid(True, alpha=0.3)
            
            plt.suptitle('Feature Distribution: Normal vs Anomalous Points', fontsize=14, y=1.02)
            plt.tight_layout()
            plt.show()

# %%
# Create comprehensive dashboard
dashboard = AnomalyDashboard(X_scaled, feature_names, y)

# Run all detectors
dashboard_results = dashboard.run_all_detectors()

# Create dashboard
dashboard_summary = dashboard.create_dashboard()

# %% [markdown]
# ## 9. Export Visualizations and Results

# %%
class VisualizationExporter:
    """Export visualizations and results"""
    
    def __init__(self, output_dir='../reports/visualizations'):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
    def export_all_visualizations(self, dashboard, df, X_scaled, feature_names, y):
        """Export all visualizations"""
        
        print("=" * 80)
        print("EXPORTING VISUALIZATIONS AND RESULTS")
        print("=" * 80)
        
        # 1. Export summary statistics
        self._export_summary_statistics(dashboard, df)
        
        # 2. Export anomaly detection results
        self._export_anomaly_results(dashboard)
        
        # 3. Export feature analysis
        self._export_feature_analysis(X_scaled, feature_names, y)
        
        # 4. Export interactive dashboards
        self._export_interactive_dashboards(dashboard, X_scaled, feature_names, y)
        
        print(f"\n🎉 All visualizations exported to: {self.output_dir}")
    
    def _export_summary_statistics(self, dashboard, df):
        """Export summary statistics"""
        
        print("\n📊 Exporting summary statistics...")
        
        summary_data = []
        
        for name, result in dashboard.results.items():
            if 'error' not in result:
                summary_data.append({
                    'method': name,
                    'anomalies_detected': result['n_anomalies'],
                    'anomaly_rate': result['anomaly_rate'],
                    'success': True
                })
            else:
                summary_data.append({
                    'method': name,
                    'anomalies_detected': 0,
                    'anomaly_rate': 0,
                    'success': False,
                    'error': result['error']
                })
        
        summary_df = pd.DataFrame(summary_data)
        summary_path = self.output_dir / 'anomaly_detection_summary.csv'
        summary_df.to_csv(summary_path, index=False)
        print(f"  ✅ Summary statistics saved: {summary_path}")
        
        # Export dataset statistics
        dataset_stats = {
            'total_samples': len(df),
            'total_features': len(feature_names),
            'fraud_rate': df['is_fraud'].mean() if 'is_fraud' in df.columns else None,
            'missing_values': df.isnull().sum().sum(),
            'data_types': str(df.dtypes.value_counts().to_dict())
        }
        
        stats_path = self.output_dir / 'dataset_statistics.json'
        with open(stats_path, 'w') as f:
            json.dump(dataset_stats, f, indent=2)
        print(f"  ✅ Dataset statistics saved: {stats_path}")
    
    def _export_anomaly_results(self, dashboard):
        """Export anomaly detection results"""
        
        print("\n🔍 Exporting anomaly results...")
        
        # Export consensus anomalies
        if hasattr(dashboard, 'consensus_anomalies'):
            consensus_df = pd.DataFrame({
                'is_anomaly': dashboard.consensus_anomalies
            })
            
            consensus_path = self.output_dir / 'consensus_anomalies.csv'
            consensus_df.to_csv(consensus_path, index=False)
            print(f"  ✅ Consensus anomalies saved: {consensus_path}")
        
        # Export individual method results
        results_dir = self.output_dir / 'method_results'
        results_dir.mkdir(exist_ok=True)
        
        for name, result in dashboard.results.items():
            if 'error' not in result:
                method_df = pd.DataFrame({
                    'is_anomaly': result['anomalies']
                })
                
                method_path = results_dir / f'{name}_anomalies.csv'
                method_df.to_csv(method_path, index=False)
        
        print(f"  ✅ Method results saved: {results_dir}")
    
    def _export_feature_analysis(self, X_scaled, feature_names, y):
        """Export feature analysis"""
        
        print("\n📈 Exporting feature analysis...")
        
        # Calculate feature importance using Isolation Forest
        iso_forest = IsolationForest(random_state=42)
        iso_forest.fit(X_scaled)
        
        if hasattr(iso_forest, 'feature_importances_'):
            importances = iso_forest.feature_importances_
            
            importance_df = pd.DataFrame({
                'feature': feature_names,
                'importance': importances
            }).sort_values('importance', ascending=False)
            
            importance_path = self.output_dir / 'feature_importance.csv'
            importance_df.to_csv(importance_path, index=False)
            print(f"  ✅ Feature importance saved: {importance_path}")
            
            # Create feature importance visualization
            fig, ax = plt.subplots(figsize=(10, 8))
            top_features = importance_df.head(20)
            ax.barh(range(len(top_features)), top_features['importance'])
            ax.set_yticks(range(len(top_features)))
            ax.set_yticklabels(top_features['feature'])
            ax.set_xlabel('Importance')
            ax.set_title('Top 20 Features for Anomaly Detection')
            ax.invert_yaxis()
            plt.tight_layout()
            
            viz_path = self.output_dir / 'feature_importance.png'
            plt.savefig(viz_path, dpi=300, bbox_inches='tight')
            print(f"  ✅ Feature importance visualization saved: {viz_path}")
            plt.close()
    
    def _export_interactive_dashboards(self, dashboard, X_scaled, feature_names, y):
        """Export interactive dashboards"""
        
        print("\n📊 Exporting interactive dashboards...")
        
        # Create interactive HTML dashboard
        import plotly.io as pio
        
        # 1. Create comparison dashboard
        if hasattr(dashboard, 'results'):
            # Prepare data for dashboard
            method_names = []
            anomaly_rates = []
            
            for name, result in dashboard.results.items():
                if 'error' not in result:
                    method_names.append(name.replace('_', ' ').title())
                    anomaly_rates.append(result['anomaly_rate'] * 100)
            
            # Create figure
            fig = go.Figure()
            
            fig.add_trace(go.Bar(
                x=method_names,
                y=anomaly_rates,
                name='Anomaly Rate',
                marker_color='indianred'
            ))
            
            fig.update_layout(
                title='Anomaly Detection Methods Comparison',
                xaxis_title='Method',
                yaxis_title='Anomaly Rate (%)',
                template='plotly_white'
            )
            
            # Save as HTML
            dashboard_path = self.output_dir / 'anomaly_dashboard.html'
            pio.write_html(fig, file=dashboard_path, auto_open=False)
            print(f"  ✅ Interactive dashboard saved: {dashboard_path}")
        
        # 2. Create 3D visualization
        if X_scaled.shape[1] >= 3:
            # Apply PCA for 3D visualization
            from sklearn.decomposition import PCA
            
            pca = PCA(n_components=3, random_state=42)
            X_3d = pca.fit_transform(X_scaled)
            
            # Get anomaly scores
            iso_forest = IsolationForest(random_state=42)
            iso_forest.fit(X_scaled)
            anomaly_scores = iso_forest.decision_function(X_scaled)
            
            # Create 3D scatter plot
            fig_3d = go.Figure(data=[go.Scatter3d(
                x=X_3d[:, 0],
                y=X_3d[:, 1],
                z=X_3d[:, 2],
                mode='markers',
                marker=dict(
                    size=3,
                    color=anomaly_scores,
                    colorscale='Viridis',
                    opacity=0.8
                )
            )])
            
            fig_3d.update_layout(
                title='3D Anomaly Visualization (PCA Projection)',
                scene=dict(
                    xaxis_title='PCA Component 1',
                    yaxis_title='PCA Component 2',
                    zaxis_title='PCA Component 3'
                )
            )
            
            # Save as HTML
            viz_3d_path = self.output_dir / '3d_anomaly_visualization.html'
            pio.write_html(fig_3d, file=viz_3d_path, auto_open=False)
            print(f"  ✅ 3D visualization saved: {viz_3d_path}")
        
        # 3. Create comprehensive report
        self._create_comprehensive_report(dashboard, X_scaled.shape)

# %%
# Export all visualizations
exporter = VisualizationExporter()
exporter.export_all_visualizations(dashboard, df, X_scaled, feature_names, y)

# %% [markdown]
# ## 10. Conclusion and Recommendations

# %%
print("=" * 80)
print("ANOMALY DETECTION VISUALIZATIONS - SUMMARY")
print("=" * 80)

print(f"\n📊 Dataset Analysis Complete:")
print(f"  • Samples analyzed: {len(df):,}")
print(f"  • Features: {len(feature_names)}")
print(f"  • Fraud rate: {df['is_fraud'].mean()*100:.2f}%" if 'is_fraud' in df.columns else "No fraud labels")

print(f"\n🔍 Anomaly Detection Methods Evaluated: {len(dashboard.results)}")

# Summary of best methods
successful_methods = []
for name, result in dashboard.results.items():
    if 'error' not in result:
        successful_methods.append({
            'name': name,
            'anomalies': result['n_anomalies'],
            'rate': result['anomaly_rate'] * 100
        })

if successful_methods:
    print(f"\n🏆 Top Performing Methods:")
    successful_methods.sort(key=lambda x: x['rate'], reverse=True)
    
    for i, method in enumerate(successful_methods[:3]):
        print(f"  {i+1}. {method['name'].replace('_', ' ').title()}:")
        print(f"     Anomalies: {method['anomalies']:,} ({method['rate']:.2f}%)")

print(f"\n🎯 Key Findings:")
print("  1. Multiple anomaly detection methods provide complementary insights")
print("  2. Statistical methods (Z-score, IQR) are good for univariate analysis")
print("  3. ML methods (Isolation Forest, LOF) work well for multivariate data")
print("  4. Clustering methods (DBSCAN, K-means) can find natural groupings")
print("  5. Consensus approaches often outperform individual methods")

print(f"\n💡 Recommendations:")
print("  1. Use ensemble of multiple anomaly detection methods")
print("  2. Focus on features with high variance and discriminative power")
print("  3. Regularly update anomaly detection models with new data")
print("  4. Combine supervised and unsupervised approaches")
print("  5. Implement real-time anomaly monitoring")

print(f"\n📈 Next Steps:")
print("  1. Deploy best anomaly detection pipeline")
print("  2. Set up automated alerting system")
print("  3. Create feedback loop for continuous improvement")
print("  4. Integrate with existing fraud detection systems")
print("  5. Monitor performance metrics regularly")

print(f"\n🎉 Anomaly detection visualizations completed successfully!")
print(f"📁 Results exported to: {exporter.output_dir}")